# Notebook to open `xmitgcm` datasets on SciServer-ceph and display basic information about them.

This code avoids OceanSpy

If the datasets fail to open, make sure that your SciServer container includes the Poseidon (ceph) and Ocean Circulation (ceph) data volumes.

TWNH Jun '26

In [1]:
import yaml
import traceback
from pathlib import Path
import re

import xmitgcm

In [2]:
# Change this to the catalog you want to test
local_catalog_file1 = '/home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/sciserver_catalogs/catalog_xmitgcm.yaml'

# If False: read one scalar from the first data variable only.
# If True: read one scalar from every data variable.
CHECK_ALL_VARS = False

In [3]:
def light_check(ds, check_all_vars=False):
    """
    Light integrity check for an xarray Dataset.

    Does not load the whole dataset. It only reads one scalar from one variable,
    or from every variable if check_all_vars=True.
    """

    print(f"    dims: {dict(ds.sizes)}")
    print(f"    data variables: {len(ds.data_vars)}")

    if len(ds.data_vars) == 0:
        print("    no data variables to test")
        return

    varnames = list(ds.data_vars) if check_all_vars else [list(ds.data_vars)[0]]

    for varname in varnames:
        da = ds[varname]

        print(f"    checking variable: {varname}, dims={da.dims}, shape={da.shape}")

        if da.size == 0:
            print("      skipping empty variable")
            continue

        indexer = {
            dim: 0
            for dim in da.dims
            if da.sizes.get(dim, 0) > 0
        }

        # Trigger a tiny actual read.
        da.isel(indexer).load()

        print("      ok")
        
def parse_range_string(value):
    """
    Convert a string like 'range(0, 1321200+1, 3600)' to a Python range.

    Only intended for trusted local catalog files.
    """
    if not isinstance(value, str):
        return value

    text = value.strip()

    if not text.startswith("range(") or not text.endswith(")"):
        return value

    inside = text[len("range("):-1]

    # This catalog contains expressions like 1321200+1.
    # Evaluate only with a restricted namespace.
    parts = [
        eval(part.strip(), {"__builtins__": {}}, {})
        for part in inside.split(",")
    ]

    return range(*parts)

def clean_xmitgcm_args(args, fast=True):
    """
    Prepare args from YAML for xmitgcm.open_mdsdataset.

    If fast=True, force opening only one iteration for speed.
    """

    args = dict(args)

    # Convert iters: "range(...)" strings into range objects.
    if "iters" in args:
        args["iters"] = parse_range_string(args["iters"])

    if fast:
        data_dir = args.get("data_dir")

        if data_dir is not None:
            available_iters = find_mitgcm_iters(data_dir, max_iters=1)

            if available_iters:
                first_iter = available_iters[0]
                args["iters"] = [first_iter]
                print(f"    fast mode: using only iters=[{first_iter}]")
            else:
                print("    fast mode: no iteration-numbered *.meta files found")
                print("    fast mode: leaving iters unchanged")

    return args

def load_xmitgcm_yaml_catalog(catalog_path):
    """
    Load a non-standard xmitgcm YAML catalog.

    This handles both forms:

      sources:
        dataset:
          ...

    and:

      dataset:
        ...

    For your file, entries are top-level.
    """

    with open(catalog_path, "r") as f:
        data = yaml.safe_load(f)

    if data is None:
        raise ValueError(f"Catalog is empty: {catalog_path}")

    if "sources" in data:
        sources = data["sources"]
    else:
        sources = data

    return sources

def open_xmitgcm_source(name, source, fast=True):
    """
    Open one YAML source using xmitgcm.open_mdsdataset.
    """

    args = source.get("args", {})
    args = clean_xmitgcm_args(args, fast=fast)

    print(f"    xmitgcm args:")
    for key, value in args.items():
        if key == "iters":
            if isinstance(value, range):
                print(
                    f"      {key}: range({value.start}, {value.stop}, {value.step}) "
                    f"[length={len(value)}]"
                )
            else:
                print(f"      {key}: {value}")
        else:
            print(f"      {key}: {value}")

    ds = xmitgcm.open_mdsdataset(**args)

    return ds

def test_xmitgcm_yaml_catalog(catalog_path, check_all_vars=False, fast=True):
    """
    Open every source in a non-standard YAML catalog using xmitgcm,
    then run a light xarray integrity check.

    If fast=True, only one available MITgcm iteration is opened per source.
    """

    sources = load_xmitgcm_yaml_catalog(catalog_path)

    passed = []
    failed = []

    for name, source in sources.items():
        print("\n" + "=" * 80)
        print(f"Testing xmitgcm source: {name}")
        print("=" * 80)

        try:
            ds = open_xmitgcm_source(name, source, fast=fast)

            light_check(ds, check_all_vars=check_all_vars)

            try:
                ds.close()
            except Exception:
                pass

            print(f"PASS: {name}")
            passed.append(name)

        except Exception as e:
            print(f"FAIL: {name}")
            print(f"Error: {e}")
            traceback.print_exc()
            failed.append((name, repr(e)))

    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Catalog: {catalog_path}")
    print(f"Fast mode: {fast}")
    print(f"Passed: {len(passed)}")
    print(f"Failed: {len(failed)}")

    if failed:
        print("\nFailed sources:")
        for name, error in failed:
            print(f"  - {name}: {error}")

    return passed, failed

def find_mitgcm_iters(data_dir, max_iters=None):
    """
    Find MITgcm iteration numbers from *.meta files in data_dir.

    Returns sorted unique iteration numbers.

    Examples matched:
      T.0000000000.meta
      U.0000132120.meta
      state.0000003600.meta
    """

    data_dir = Path(data_dir)

    iters = set()

    pattern = re.compile(r"\.(\d{10})\.meta$")

    for path in data_dir.glob("*.meta"):
        match = pattern.search(path.name)
        if match:
            iters.add(int(match.group(1)))

            if max_iters is not None and len(iters) >= max_iters:
                break

    return sorted(iters)

In [4]:
passed, failed = test_xmitgcm_yaml_catalog(
    local_catalog_file1,
    check_all_vars=CHECK_ALL_VARS,
    fast=True,
)


Testing xmitgcm source: KangerFjord
    fast mode: using only iters=[2023920]
    xmitgcm args:
      data_dir: /home/idies/workspace/ocean_circulation_ceph/fromNeil
      grid_dir: /home/idies/workspace/ocean_circulation_ceph/fromNeil
      delta_t: 5
      ref_date: 2007-08-21:30
      grid_vars_to_coords: False
      ignore_unknown_vars: True
      iters: [2023920]


/home/idies/mambaforge/envs/Oceanography/lib/python3.12/site-packages/xmitgcm/mds_store.py:927: UserWarning: Couldn't find available_diagnostics.log in /home/idies/workspace/ocean_circulation_ceph/fromNeil or /home/idies/workspace/ocean_circulation_ceph/fromNeil. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "


    dims: {'time': 1, 'XC': 240, 'YC': 240, 'XG': 240, 'YG': 240, 'Z': 100, 'Zp1': 101, 'Zu': 100, 'Zl': 100}
    data variables: 35
    checking variable: iter, dims=('time',), shape=(1,)
      ok
PASS: KangerFjord

Testing xmitgcm source: EGshelfSJsec500m_3H_hydro
    fast mode: using only iters=[149400]
    xmitgcm args:
      data_dir: /home/idies/workspace/ocean_circulation_ceph/fromMarcello/exp1/all_result/3H/
      grid_dir: /home/idies/workspace/ocean_circulation_ceph/fromMarcello/exp1/all_result
      delta_t: 6
      ref_date: 2003-06-01
      grid_vars_to_coords: False
      ignore_unknown_vars: True
      iters: [149400]
    dims: {'time': 1, 'XC': 725, 'YC': 605, 'XG': 725, 'YG': 605, 'Z': 97, 'Zp1': 98, 'Zu': 97, 'Zl': 97}
    data variables: 26
    checking variable: iter, dims=('time',), shape=(1,)
      ok
PASS: EGshelfSJsec500m_3H_hydro

Testing xmitgcm source: fldEGshelfSJsec500m_6H_hydro
    fast mode: using only iters=[149400]
    xmitgcm args:
      data_dir: /hom

/home/idies/mambaforge/envs/Oceanography/lib/python3.12/site-packages/xmitgcm/mds_store.py:927: UserWarning: Couldn't find available_diagnostics.log in /home/idies/workspace/ocean_circulation_ceph/fromMarcello/exp6/all_result/3H/ or /home/idies/workspace/ocean_circulation_ceph/fromMarcello/exp6/all_result. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "


    dims: {'time': 1, 'XC': 725, 'YC': 605, 'XG': 725, 'YG': 605, 'Z': 97, 'Zp1': 98, 'Zu': 97, 'Zl': 97}
    data variables: 26
    checking variable: iter, dims=('time',), shape=(1,)
      ok
PASS: EGshelfSJsec500m_3H_NONhydro

Testing xmitgcm source: fldEGshelfSJsec500m_6H_NONhydro
    fast mode: using only iters=[149400]
    xmitgcm args:
      data_dir: /home/idies/workspace/ocean_circulation_ceph/fromMarcello/exp6/all_result/3H/
      grid_dir: /home/idies/workspace/ocean_circulation_ceph/fromMarcello/exp6/all_result
      delta_t: 6
      iters: [149400]
      ref_date: 2003-06-01
      grid_vars_to_coords: False
      ignore_unknown_vars: True
    dims: {'time': 1, 'XC': 725, 'YC': 605, 'XG': 725, 'YG': 605, 'Z': 97, 'Zp1': 98, 'Zu': 97, 'Zl': 97}
    data variables: 26
    checking variable: iter, dims=('time',), shape=(1,)
      ok
PASS: fldEGshelfSJsec500m_6H_NONhydro

Testing xmitgcm source: exfEGshelfSJsec500m_6H_NONhydro


/home/idies/mambaforge/envs/Oceanography/lib/python3.12/site-packages/xmitgcm/mds_store.py:927: UserWarning: Couldn't find available_diagnostics.log in /home/idies/workspace/ocean_circulation_ceph/fromMarcello/exp6/all_result/3H/ or /home/idies/workspace/ocean_circulation_ceph/fromMarcello/exp6/all_result. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "


    fast mode: using only iters=[824400]
    xmitgcm args:
      data_dir: /home/idies/workspace/ocean_circulation_ceph/fromMarcello/exp6/all_result/6H/
      grid_dir: /home/idies/workspace/ocean_circulation_ceph/fromMarcello/exp6/all_result
      delta_t: 6
      ref_date: 2003-06-01
      grid_vars_to_coords: False
      ignore_unknown_vars: True
      iters: [824400]


/home/idies/mambaforge/envs/Oceanography/lib/python3.12/site-packages/xmitgcm/mds_store.py:927: UserWarning: Couldn't find available_diagnostics.log in /home/idies/workspace/ocean_circulation_ceph/fromMarcello/exp6/all_result/6H/ or /home/idies/workspace/ocean_circulation_ceph/fromMarcello/exp6/all_result. Using default version.
  warnings.warn("Couldn't find available_diagnostics.log "


    dims: {'time': 1, 'XC': 725, 'YC': 605, 'XG': 725, 'YG': 605, 'Z': 97, 'Zp1': 98, 'Zu': 97, 'Zl': 97}
    data variables: 29
    checking variable: iter, dims=('time',), shape=(1,)
      ok
PASS: exfEGshelfSJsec500m_6H_NONhydro

SUMMARY
Catalog: /home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/sciserver_catalogs/catalog_xmitgcm.yaml
Fast mode: True
Passed: 7
Failed: 0


In [5]:
import pandas as pd

results = pd.DataFrame(
    [{"source": name, "status": "passed", "error": ""} for name in passed]
    +
    [{"source": name, "status": "failed", "error": error} for name, error in failed]
)

results

,source,status,error
0,KangerFjord,passed,
1,EGshelfSJsec500m_3H_hydro,passed,
2,fldEGshelfSJsec500m_6H_hydro,passed,
3,exfEGshelfSJsec500m_6H_hydro,passed,
4,EGshelfSJsec500m_3H_NONhydro,passed,
5,fldEGshelfSJsec500m_6H_NONhydro,passed,
6,exfEGshelfSJsec500m_6H_NONhydro,passed,
